In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# PyRPL initialization
# ============================================================

from pyrpl import Pyrpl

HOSTNAME = "rp-f0c970.local"

print("=" * 60)
print("INITIALIZING PyRPL")
print("=" * 60)

print("Connecting to Red Pitaya...")

p = Pyrpl(
    config="my_config",
    hostname=HOSTNAME,
    reloadfpga=True,
    gui=False,
)

rp_pyrpl = p.rp

print("PyRPL connected.")
print()
print("Available PyRPL modules:")

print(f"  ASG0:  {rp_pyrpl.asg0}")
print(f"  ASG1:  {rp_pyrpl.asg1}")
print(f"  PID0:  {rp_pyrpl.pid0}")
print(f"  PID1:  {rp_pyrpl.pid1}")
print(f"  IQ0:   {rp_pyrpl.iq0}")
print(f"  IIR:   {rp_pyrpl.iir}")


# ============================================================
# Low-level Red Pitaya API
#
# IMPORTANT:
# We use the same rp API as REAL_BUILD for the scans.
# PyRPL has already loaded the FPGA image above.
# ============================================================

import rp

print()
print("Initializing low-level Red Pitaya API...")

rp.rp_Init()

print("Low-level API initialized.")


# ============================================================
# Hardware configuration
# ============================================================

N = 16384
ADC_CLK = 125e6

GENERATOR_CHANNEL = rp.RP_CH_1

# IN1 = photodiode
PHOTODIODE_CHANNEL = rp.RP_CH_1

# IN2 = measured laser modulation voltage
MODULATION_CHANNEL = rp.RP_CH_2


# ============================================================
# OUT1 calibration from REAL_BUILD
# ============================================================

CAL_MIN_V = 0.000488
CAL_MAX_V = 1.037964
CAL_SCALE_V = CAL_MAX_V - CAL_MIN_V


# ============================================================
# Supported acquisition decimations
# ============================================================

VALID_DECIMATIONS = [
    1,
    8,
    64,
    1024,
    8192,
    65536,
]

DECIMATION_MAP = {
    1: rp.RP_DEC_1,
    8: rp.RP_DEC_8,
    64: rp.RP_DEC_64,
    1024: rp.RP_DEC_1024,
    8192: rp.RP_DEC_8192,
    65536: rp.RP_DEC_65536,
}


# ============================================================
# Helper functions
# ============================================================

def voltage_to_normalized(voltage):
    """
    Convert physical voltage into the normalized arbitrary
    waveform representation used by the Red Pitaya generator.
    """

    normalized = (
        voltage - CAL_MIN_V
    ) / CAL_SCALE_V

    if np.any(normalized < -1.0) or np.any(normalized > 1.0):
        raise ValueError(
            "Requested voltage exceeds calibrated output range."
        )

    return np.clip(
        normalized,
        -1.0,
        1.0,
    )


def make_voltage_ramp(
    start_voltage,
    end_voltage,
):
    """
    Create an N-point linear voltage ramp.
    """

    voltage = np.linspace(
        start_voltage,
        end_voltage,
        N,
    )

    normalized = voltage_to_normalized(
        voltage
    )

    waveform = rp.arbBuffer(N)

    for i in range(N):
        waveform[i] = float(
            normalized[i]
        )

    return waveform


def configure_voltage_ramp(
    start_voltage,
    end_voltage,
    duration_s,
):
    """
    Configure the Red Pitaya generator for a single
    arbitrary-waveform burst from start_voltage to
    end_voltage.
    """

    waveform = make_voltage_ramp(
        start_voltage,
        end_voltage,
    )

    rp.rp_GenWaveform(
        GENERATOR_CHANNEL,
        rp.RP_WAVEFORM_ARBITRARY,
    )

    rp.rp_GenArbWaveform(
        GENERATOR_CHANNEL,
        waveform.cast(),
        N,
    )

    rp.rp_GenFreqDirect(
        GENERATOR_CHANNEL,
        1.0 / duration_s,
    )

    rp.rp_GenAmp(
        GENERATOR_CHANNEL,
        1.0,
    )

    rp.rp_GenOffset(
        GENERATOR_CHANNEL,
        0.0,
    )

    rp.rp_GenPhase(
        GENERATOR_CHANNEL,
        0.0,
    )

    # One-shot burst.
    rp.rp_GenMode(
        GENERATOR_CHANNEL,
        rp.RP_GEN_MODE_BURST,
    )

    rp.rp_GenBurstCount(
        GENERATOR_CHANNEL,
        1,
    )

    rp.rp_GenBurstRepetitions(
        GENERATOR_CHANNEL,
        1,
    )

    # These are the important endpoint controls from
    # the working REAL_BUILD notebook.
    rp.rp_GenSetInitGenValue(
        GENERATOR_CHANNEL,
        start_voltage,
    )

    rp.rp_GenBurstLastValue(
        GENERATOR_CHANNEL,
        end_voltage,
    )

    rp.rp_GenTriggerSource(
        GENERATOR_CHANNEL,
        rp.RP_GEN_TRIG_SRC_INTERNAL,
    )

    rp.rp_GenOutEnableSync(True)


def move_voltage(
    start_voltage,
    end_voltage,
    duration_s,
):
    """
    Move OUT1 smoothly from one voltage to another.

    This is NOT acquired by the scope.
    """

    configure_voltage_ramp(
        start_voltage,
        end_voltage,
        duration_s,
    )

    rp.rp_GenTriggerOnly(
        GENERATOR_CHANNEL,
    )

    time.sleep(
        duration_s + 0.05
    )


def configure_acquisition(
    decimation_enum,
):
    """
    Configure acquisition exactly as in REAL_BUILD.
    """

    rp.rp_AcqReset()

    rp.rp_AcqSetDecimation(
        decimation_enum
    )

    # 8192 samples pre-trigger.
    rp.rp_AcqSetTriggerDelay(
        8192
    )

    # Generator positive edge triggers acquisition.
    rp.rp_AcqSetTriggerSrc(
        rp.RP_TRIG_SRC_AWG_PE
    )


def choose_decimation(
    sweep_duration_s,
):
    """
    Select the smallest supported decimation that gives
    enough acquisition time for the sweep.
    """

    required_decimation = (
        sweep_duration_s
        * ADC_CLK
        / N
    )

    target = next(
        (
            d
            for d in VALID_DECIMATIONS
            if d >= required_decimation
        ),
        None,
    )

    if target is None:
        raise ValueError(
            "Sweep is too slow for the 16,384-sample buffer."
        )

    return target


print()
print("=" * 60)
print("INITIALIZATION COMPLETE")
print("=" * 60)
print()
print(f"ASG waveform memory: {N} points")
print("PyRPL FPGA image loaded.")
print("Low-level RP scan functions ready.")
print()
print("The PyRPL object 'p' remains available for later cells.")
print("For example:")
print("  p.rp.iq0")
print("  p.rp.pid0")
print("  p.rp.asg0")

INFO:pyrpl:All your PyRPL settings will be saved to the config file
    /Users/brandon/pyrpl_user_dir/config/my_config.yml
If you would like to restart PyRPL with these settings, type "pyrpl.exe my_config" in a windows terminal or 
    from pyrpl import Pyrpl
    p = Pyrpl('my_config')
in a python terminal.


INITIALIZING PyRPL
Connecting to Red Pitaya...


INFO:pyrpl.redpitaya:Found FPGA binfile at: /Users/brandon/Documents/Python Scripts/RedPitaya/.venv/lib/python3.13/site-packages/pyrpl/fpga/red_pitaya.bin
INFO:pyrpl.redpitaya:Found DTBO file at: /Users/brandon/Documents/Python Scripts/RedPitaya/.venv/lib/python3.13/site-packages/pyrpl/fpga/red_pitaya.dtbo
INFO:pyrpl.redpitaya:Successfully connected to Redpitaya with hostname rp-f0c970.local.


PyRPL connected.

Available PyRPL modules:
  ASG0:  <pyrpl.hardware_modules.asg.make_asg.<locals>.Asg object at 0x111ba9010>
  ASG1:  <pyrpl.hardware_modules.asg.make_asg.<locals>.Asg object at 0x111ba9160>
  PID0:  <pyrpl.hardware_modules.pid.Pid object at 0x111ba9550>
  PID1:  <pyrpl.hardware_modules.pid.Pid object at 0x111bb4190>
  IQ0:   <pyrpl.hardware_modules.iq.Iq object at 0x111ba9400>
  IIR:   <pyrpl.hardware_modules.iir.iir.IIR object at 0x111ba97f0>


ModuleNotFoundError: No module named 'rp'

In [ ]:
# ============================================================
# CELL 2 — COARSE LASER SWEEP
# ============================================================

SCAN_AMPLITUDE_V = 0.8
CENTER_VOLTAGE = 0.0

SWEEP_DURATION_S = 8.589
SETTLE_TIME_S = 10.0

TRANSITION_TIME_S = 3.0

# Physical OUT1 voltage before this scan begins.
#
# Cell 1 starts at 0 V.
STARTING_VOLTAGE = 0.0


# ============================================================
# SCAN LIMITS
# ============================================================

SCAN_START_V = (
    CENTER_VOLTAGE
    - SCAN_AMPLITUDE_V / 2
)

SCAN_END_V = (
    CENTER_VOLTAGE
    + SCAN_AMPLITUDE_V / 2
)


if SCAN_START_V < -1.1 or SCAN_END_V > 1.1:
    raise ValueError(
        f"Requested scan "
        f"{SCAN_START_V:.3f} -> "
        f"{SCAN_END_V:.3f} V is outside "
        f"the expected Red Pitaya range."
    )


# ============================================================
# ACQUISITION PARAMETERS
# ============================================================

TARGET_DEC = choose_decimation(
    SWEEP_DURATION_S
)

DEC_ENUM = DECIMATION_MAP[TARGET_DEC]

FS = ADC_CLK / TARGET_DEC

BUFFER_DURATION_S = N / FS

SWEEP_FREQUENCY_HZ = (
    1.0 / SWEEP_DURATION_S
)

PRE_TRIG_WAIT_S = (
    (N / 2) / FS
    + 0.1
)


# ============================================================
# PRINT CONFIGURATION
# ============================================================

print()
print("=" * 60)
print("COARSE LASER SWEEP")
print("=" * 60)

print()
print("Scan:")
print(
    f"  Start:             "
    f"{SCAN_START_V:+.6f} V"
)

print(
    f"  End:               "
    f"{SCAN_END_V:+.6f} V"
)

print(
    f"  Width:             "
    f"{SCAN_AMPLITUDE_V:.6f} V"
)

print(
    f"  Center:            "
    f"{CENTER_VOLTAGE:+.6f} V"
)

print()
print("Timing:")
print(
    f"  Sweep:             "
    f"{SWEEP_DURATION_S:.3f} s"
)

print(
    f"  Frequency:         "
    f"{SWEEP_FREQUENCY_HZ:.6f} Hz"
)

print(
    f"  Starting voltage:  "
    f"{STARTING_VOLTAGE:+.6f} V"
)

print(
    f"  Transition:        "
    f"{TRANSITION_TIME_S:.3f} s"
)

print(
    f"  Settle:            "
    f"{SETTLE_TIME_S:.3f} s"
)

print()
print("Acquisition:")
print(
    f"  Decimation:        "
    f"{TARGET_DEC}"
)

print(
    f"  Sample rate:       "
    f"{FS:.3f} Hz"
)

print(
    f"  Buffer duration:   "
    f"{BUFFER_DURATION_S:.3f} s"
)


# ============================================================
# BUFFERS
# ============================================================

fbuff_photodiode = rp.fBuffer(N)
fbuff_modulation = rp.fBuffer(N)


try:

    # ========================================================
    # MOVE TO SCAN START
    # ========================================================

    print()
    print(
        f"Moving smoothly: "
        f"{STARTING_VOLTAGE:+.6f} -> "
        f"{SCAN_START_V:+.6f} V"
    )

    if not np.isclose(
        STARTING_VOLTAGE,
        SCAN_START_V,
    ):

        move_voltage(
            STARTING_VOLTAGE,
            SCAN_START_V,
            TRANSITION_TIME_S,
        )

    else:

        print("Already at scan starting voltage.")


    # ========================================================
    # SETTLE
    # ========================================================

    print()
    print(
        f"Holding at "
        f"{SCAN_START_V:+.6f} V "
        f"for {SETTLE_TIME_S:.1f} s..."
    )

    time.sleep(
        SETTLE_TIME_S
    )


    # ========================================================
    # CONFIGURE ACQUISITION
    # ========================================================

    print()
    print("Starting acquisition system...")

    configure_acquisition(
        DEC_ENUM
    )

    rp.rp_AcqStart()

    time.sleep(
        PRE_TRIG_WAIT_S
    )


    # ========================================================
    # CONFIGURE SWEEP
    # ========================================================

    configure_voltage_ramp(
        SCAN_START_V,
        SCAN_END_V,
        SWEEP_DURATION_S,
    )


    # ========================================================
    # START SWEEP
    # ========================================================

    print("Starting coarse sweep...")

    rp.rp_GenTriggerOnly(
        GENERATOR_CHANNEL
    )


    # ========================================================
    # WAIT FOR ACQUISITION TRIGGER
    # ========================================================

    deadline = (
        time.time()
        + 5.0
    )

    while True:

        trigger_state = (
            rp.rp_AcqGetTriggerState()[1]
        )

        if (
            trigger_state
            == rp.RP_TRIG_STATE_TRIGGERED
        ):
            break

        if time.time() > deadline:
            raise RuntimeError(
                "Timed out waiting for acquisition trigger."
            )

        time.sleep(
            0.001
        )


    # ========================================================
    # WAIT FOR FULL BUFFER
    # ========================================================

    deadline = (
        time.time()
        + BUFFER_DURATION_S
        + 2.0
    )

    while True:

        buffer_filled = (
            rp.rp_AcqGetBufferFillState()[1]
        )

        if buffer_filled:
            break

        if time.time() > deadline:
            raise RuntimeError(
                "Timed out waiting for acquisition buffer."
            )

        time.sleep(
            0.001
        )


    # ========================================================
    # READ PHOTODIODE — IN1
    # ========================================================

    rp.rp_AcqGetOldestDataV(
        PHOTODIODE_CHANNEL,
        N,
        fbuff_photodiode,
    )

    coarse_photodiode = np.array(
        [
            fbuff_photodiode[i]
            for i in range(N)
        ],
        dtype=float,
    )


    # ========================================================
    # READ ACTUAL MODULATION — IN2
    # ========================================================

    rp.rp_AcqGetOldestDataV(
        MODULATION_CHANNEL,
        N,
        fbuff_modulation,
    )

    coarse_modulation = np.array(
        [
            fbuff_modulation[i]
            for i in range(N)
        ],
        dtype=float,
    )


    # ========================================================
    # TIME AXIS
    # ========================================================

    coarse_time = (
        np.arange(N) / FS
    )


    # ========================================================
    # RESULTS
    # ========================================================

    print()
    print("=" * 60)
    print("COARSE SWEEP COMPLETE")
    print("=" * 60)

    print()
    print(
        f"Measured modulation: "
        f"{np.min(coarse_modulation):.6f} -> "
        f"{np.max(coarse_modulation):.6f} V"
    )

    print(
        f"Photodiode: "
        f"{np.min(coarse_photodiode):.6f} -> "
        f"{np.max(coarse_photodiode):.6f} V"
    )


    # ========================================================
    # PLOT
    # ========================================================

    plt.figure(
        figsize=(12, 7)
    )

    plt.plot(
        coarse_modulation,
        coarse_photodiode,
        linewidth=1.2,
    )

    plt.xlabel(
        "Laser Modulation Voltage (V)"
    )

    plt.ylabel(
        "Photodiode Signal (V)"
    )

    plt.title(
        "Coarse Laser Modulation Sweep"
    )

    plt.grid(
        True,
        linestyle="--",
        alpha=0.7,
    )

    plt.tight_layout()
    plt.show()


    print()
    print("Available variables:")
    print("  coarse_time")
    print("  coarse_modulation")
    print("  coarse_photodiode")


except Exception:

    print()
    print("Coarse scan failed.")

    raise

In [ ]:
# ============================================================
# CELL 3 — FINE LASER SWEEP
# ============================================================

# Center this around the peak found from the coarse scan.
CENTER_VOLTAGE = 0.05

# Total width of fine scan.
#
# Example:
#   0.10 V -> center ± 0.05 V
#
SCAN_AMPLITUDE_V = 0.10

SWEEP_DURATION_S = 1.0
SETTLE_TIME_S = 5.0

TRANSITION_TIME_S = 3.0

# This should be the voltage where Cell 2 finished.
#
# For example, if Cell 2 was:
#
#   -0.4 -> +0.4 V
#
# use:
#
#   STARTING_VOLTAGE = +0.4
#
STARTING_VOLTAGE = 0.4


# ============================================================
# SCAN LIMITS
# ============================================================

SCAN_START_V = (
    CENTER_VOLTAGE
    - SCAN_AMPLITUDE_V / 2
)

SCAN_END_V = (
    CENTER_VOLTAGE
    + SCAN_AMPLITUDE_V / 2
)


if SCAN_START_V < -1.1 or SCAN_END_V > 1.1:
    raise ValueError(
        f"Requested scan "
        f"{SCAN_START_V:.3f} -> "
        f"{SCAN_END_V:.3f} V is outside "
        f"the expected Red Pitaya range."
    )


# ============================================================
# ACQUISITION
# ============================================================

TARGET_DEC = choose_decimation(
    SWEEP_DURATION_S
)

DEC_ENUM = DECIMATION_MAP[TARGET_DEC]

FS = ADC_CLK / TARGET_DEC

BUFFER_DURATION_S = N / FS

SWEEP_FREQUENCY_HZ = (
    1.0 / SWEEP_DURATION_S
)

PRE_TRIG_WAIT_S = (
    (N / 2) / FS
    + 0.1
)


# ============================================================
# PRINT CONFIGURATION
# ============================================================

print()
print("=" * 60)
print("FINE LASER SWEEP")
print("=" * 60)

print()
print("Scan:")
print(
    f"  Start:             "
    f"{SCAN_START_V:+.6f} V"
)

print(
    f"  End:               "
    f"{SCAN_END_V:+.6f} V"
)

print(
    f"  Width:             "
    f"{SCAN_AMPLITUDE_V:.6f} V"
)

print(
    f"  Center:            "
    f"{CENTER_VOLTAGE:+.6f} V"
)

print()
print("Timing:")
print(
    f"  Sweep:             "
    f"{SWEEP_DURATION_S:.3f} s"
)

print(
    f"  Frequency:         "
    f"{SWEEP_FREQUENCY_HZ:.6f} Hz"
)

print(
    f"  Starting voltage:  "
    f"{STARTING_VOLTAGE:+.6f} V"
)

print(
    f"  Transition:        "
    f"{TRANSITION_TIME_S:.3f} s"
)

print(
    f"  Settle:            "
    f"{SETTLE_TIME_S:.3f} s"
)

print()
print("Acquisition:")
print(
    f"  Decimation:        "
    f"{TARGET_DEC}"
)

print(
    f"  Sample rate:       "
    f"{FS:.3f} Hz"
)

print(
    f"  Buffer duration:   "
    f"{BUFFER_DURATION_S:.3f} s"
)


# ============================================================
# BUFFERS
# ============================================================

fbuff_photodiode = rp.fBuffer(N)
fbuff_modulation = rp.fBuffer(N)


try:

    # ========================================================
    # MOVE FROM PREVIOUS SCAN END TO FINE SCAN START
    # ========================================================

    print()
    print(
        f"Moving smoothly: "
        f"{STARTING_VOLTAGE:+.6f} -> "
        f"{SCAN_START_V:+.6f} V"
    )

    if not np.isclose(
        STARTING_VOLTAGE,
        SCAN_START_V,
    ):

        move_voltage(
            STARTING_VOLTAGE,
            SCAN_START_V,
            TRANSITION_TIME_S,
        )

    else:

        print("Already at fine-scan starting voltage.")


    # ========================================================
    # SETTLE
    # ========================================================

    print()
    print(
        f"Holding at "
        f"{SCAN_START_V:+.6f} V "
        f"for {SETTLE_TIME_S:.1f} s..."
    )

    time.sleep(
        SETTLE_TIME_S
    )


    # ========================================================
    # CONFIGURE ACQUISITION
    # ========================================================

    print()
    print("Starting acquisition system...")

    configure_acquisition(
        DEC_ENUM
    )

    rp.rp_AcqStart()

    time.sleep(
        PRE_TRIG_WAIT_S
    )


    # ========================================================
    # CONFIGURE FINE SWEEP
    # ========================================================

    configure_voltage_ramp(
        SCAN_START_V,
        SCAN_END_V,
        SWEEP_DURATION_S,
    )


    # ========================================================
    # START FINE SWEEP
    # ========================================================

    print("Starting fine sweep...")

    rp.rp_GenTriggerOnly(
        GENERATOR_CHANNEL
    )


    # ========================================================
    # WAIT FOR ACQUISITION TRIGGER
    # ========================================================

    deadline = (
        time.time()
        + 5.0
    )

    while True:

        trigger_state = (
            rp.rp_AcqGetTriggerState()[1]
        )

        if (
            trigger_state
            == rp.RP_TRIG_STATE_TRIGGERED
        ):
            break

        if time.time() > deadline:
            raise RuntimeError(
                "Timed out waiting for acquisition trigger."
            )

        time.sleep(
            0.001
        )


    # ========================================================
    # WAIT FOR BUFFER
    # ========================================================

    deadline = (
        time.time()
        + BUFFER_DURATION_S
        + 2.0
    )

    while True:

        buffer_filled = (
            rp.rp_AcqGetBufferFillState()[1]
        )

        if buffer_filled:
            break

        if time.time() > deadline:
            raise RuntimeError(
                "Timed out waiting for acquisition buffer."
            )

        time.sleep(
            0.001
        )


    # ========================================================
    # READ PHOTODIODE — IN1
    # ========================================================

    rp.rp_AcqGetOldestDataV(
        PHOTODIODE_CHANNEL,
        N,
        fbuff_photodiode,
    )

    fine_photodiode = np.array(
        [
            fbuff_photodiode[i]
            for i in range(N)
        ],
        dtype=float,
    )


    # ========================================================
    # READ ACTUAL MODULATION — IN2
    # ========================================================

    rp.rp_AcqGetOldestDataV(
        MODULATION_CHANNEL,
        N,
        fbuff_modulation,
    )

    fine_modulation = np.array(
        [
            fbuff_modulation[i]
            for i in range(N)
        ],
        dtype=float,
    )


    # ========================================================
    # TIME
    # ========================================================

    fine_time = (
        np.arange(N) / FS
    )


    # ========================================================
    # FIND PEAK
    # ========================================================

    peak_index = np.argmax(
        fine_photodiode
    )

    peak_voltage = (
        fine_modulation[peak_index]
    )

    peak_signal = (
        fine_photodiode[peak_index]
    )


    # ========================================================
    # RESULTS
    # ========================================================

    print()
    print("=" * 60)
    print("FINE SWEEP COMPLETE")
    print("=" * 60)

    print()
    print(
        f"Measured modulation: "
        f"{np.min(fine_modulation):.6f} -> "
        f"{np.max(fine_modulation):.6f} V"
    )

    print(
        f"Photodiode: "
        f"{np.min(fine_photodiode):.6f} -> "
        f"{np.max(fine_photodiode):.6f} V"
    )

    print()
    print("Measured peak:")
    print(
        f"  Voltage: "
        f"{peak_voltage:+.6f} V"
    )

    print(
        f"  Signal:  "
        f"{peak_signal:.6f} V"
    )


    # ========================================================
    # PLOT
    # ========================================================

    plt.figure(
        figsize=(12, 7)
    )

    plt.plot(
        fine_modulation,
        fine_photodiode,
        linewidth=1.2,
        label="Photodiode",
    )

    plt.axvline(
        CENTER_VOLTAGE,
        linestyle="--",
        linewidth=1,
        label=(
            f"Scan center = "
            f"{CENTER_VOLTAGE:+.4f} V"
        ),
    )

    plt.axvline(
        peak_voltage,
        linestyle=":",
        linewidth=1,
        label=(
            f"Measured peak = "
            f"{peak_voltage:+.4f} V"
        ),
    )

    plt.xlabel(
        "Laser Modulation Voltage (V)"
    )

    plt.ylabel(
        "Photodiode Signal (V)"
    )

    plt.title(
        "Fine Laser Modulation Sweep"
    )

    plt.grid(
        True,
        linestyle="--",
        alpha=0.7,
    )

    plt.legend()

    plt.tight_layout()
    plt.show()


    print()
    print("Available variables:")
    print("  fine_time")
    print("  fine_modulation")
    print("  fine_photodiode")
    print("  peak_voltage")
    print("  peak_signal")


except Exception:

    print()
    print("Fine scan failed.")

    raise